In [ ]:
import pickle
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scanpy as sc
from scipy.stats import norm

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.PerSamplePathwayAnalysis.pathway_convergence import (
    load_symbol_vocab, overlap_enrichment, pairwise_overlap,
)

ZDIR = config.ZSCORES_MIXED_DIR
PCDIR = config.PATHWAY_CONV_DIR
PFDIR = config.PATHWAY_CONV_FIG_DIR
PCDIR.mkdir(parents=True, exist_ok=True)
PFDIR.mkdir(parents=True, exist_ok=True)
PP = config.PATHWAY_CONV_PARAMS

universe_syms, sym2idx, col2sym = load_symbol_vocab(None)  # phenotype-independent, for gene-level naming


def overlap_stats(masks, n_perm=200, seed=42):
    e = overlap_enrichment(masks, n_perm=n_perm, seed=seed)
    jacc, _ = pairwise_overlap(masks)
    e['nonzero_pair_frac'] = float((jacc > 0).mean())
    e['n_sig_median'] = float(np.median(masks.sum(axis=1)))
    return e

In [ ]:
# 1. Gene-level and pathway-level reoccurrence per phenotype, across BH-FDR q = [0.05, 0.10, 0.15,
# 0.20, 0.25] (same grid as 3_disease_scoring.ipynb sec. 6) -- does the same deviation converge onto
# shared genes/pathways across patients within a cohort, more than a size-matched random draw would
# (Wolfers 2018 JAMA Psych / Segal 2023 Nat Neurosci deviation-overlap design)?
#
# Thin runner: all per-sample identity (which genes/pathways/up/down recurred, in which patients) is
# computed and cached by run_pathway_reoccurrence_sweep.py -> PerSamplePathwayAnalysis/<slug>/
# reoccurrence<q-tag>.pkl (mirrors run_pathway_convergence_batch.py's own sig.pkl/sig_directional.pkl
# storage). Run that script first. This cell only loads those files and aggregates/plots.
from MixedEffectsModeling.PerSamplePathwayAnalysis.pathway_convergence import q_tag

QS = [0.05, 0.10, 0.15, 0.20, 0.25]


def recurring_items(mask, names, study, level, q):
    n_pat = mask.shape[0]
    counts = mask.sum(axis=0)
    idx = np.where(counts > 0)[0]
    return pd.DataFrame({
        'study_dir': study, 'level': level, 'q': q,
        'item': [names[i] for i in idx], 'n_patients_sig': counts[idx], 'n_pat': n_pat,
        'rate': counts[idx] / n_pat,
    }).sort_values('n_patients_sig', ascending=False)


sweep_rows, detail_rows = [], []
for pdir in sorted(d for d in PCDIR.iterdir() if d.is_dir() and (d / 'sig.pkl').exists()):
    for q in QS:
        rec_path = pdir / f'reoccurrence{q_tag(q)}.pkl'
        if not rec_path.exists():
            print(f'{pdir.name} q={q}: missing (run run_pathway_reoccurrence_sweep.py)', flush=True)
            continue
        rec = pickle.load(open(rec_path, 'rb'))
        n_pat = rec['gene_sig'].shape[0]
        if n_pat < 3:
            continue

        for level, names in [('gene', universe_syms), ('path_ORA', rec['terms']),
                             ('path_up_ORA', rec['terms']), ('path_down_ORA', rec['terms'])]:
            key = {'gene': 'gene_sig', 'path_ORA': 'path_sig',
                  'path_up_ORA': 'path_sig_up', 'path_down_ORA': 'path_sig_down'}[level]
            if key not in rec:
                continue
            mask = rec[key]
            sweep_rows.append(dict(study_dir=pdir.name, level=level, q=q, n_pat=n_pat,
                                   **overlap_stats(mask, n_perm=100 if level == 'gene' else 200)))
            detail_rows.append(recurring_items(mask, names, pdir.name, level, q))

sweep = pd.DataFrame(sweep_rows)
detail = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()
sweep.to_csv(PCDIR / 'gene_pathway_reoccurrence_sweep.csv', index=False)
detail.to_csv(PCDIR / 'gene_pathway_reoccurrence_detail.csv', index=False)
display(sweep)
print(f'\n{len(detail)} recurring (study, level, q, item) rows -> gene_pathway_reoccurrence_detail.csv')
print(detail.sort_values('n_patients_sig', ascending=False).head(20).to_string(index=False))

from MixedEffectsModeling.Benchmark import db_hit_compare as dc

CROSS_PATH = config.ROOT / 'MixedEffectsModeling' / 'Benchmark' / 'cross_study_replication.csv'
if CROSS_PATH.exists():
    cross_df = pd.read_csv(CROSS_PATH)
else:
    cross_df = dc.cross_study_replication()
    cross_df.to_csv(CROSS_PATH, index=False)

display(cross_df)